# Final Submission — EXP-020 + EXP-028 + EXP-032 Rank Average

대회 코드 제출용 통합 노트북. 세 파이프라인을 순차 실행 후 최종 Rank Average 앙상블.

| 파트 | 실험 | 방식 | 피처 |
|---|---|---|---|
| A | EXP-020 | LGB+CAT+XGB, seed=42, Rank Avg | FE-v1 (85개) |
| B | EXP-028 | LGB+CAT+XGB, 5 seeds, Rank Avg | FE-v1 (85개) |
| C | EXP-032 | LGB(FE-v1)+CAT(FE-v2+TE)+XGB(FE-v2+TE+ITE) | mixed |
| Final | — | Rank Average(A, B, C) | — |

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score, accuracy_score
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import xgboost as xgb

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
OUT_DIR  = Path('../data/submissions')
DOCS_DIR = Path('../docs')
TARGET   = '임신 성공 여부'
N_FOLDS  = 5
SEED     = 42
AUTHOR   = '조여진'

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'train: {train.shape}  /  test: {test.shape}')

train: (256351, 69)  /  test: (90067, 68)


## 공통 피처 준비

In [2]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features_v1(df):
    """FE-v1 (85개) — EXP-020/028 기준"""
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']    = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부']  = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']      = df[male_cols].sum(axis=1)
    df['여성_불임_합계']      = df[female_cols].sum(axis=1)
    df['부부_불임_합계']      = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']      = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']      = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']    = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']  = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    return df

def add_derived_features_v2(df):
    """FE-v2 (89개) — v1 + 4개 추가, EXP-032 CAT/XGB용"""
    df = add_derived_features_v1(df)
    eps = 1e-6
    df['임신_성공률']   = df['총 임신 횟수'] / (df['총 시술 횟수'] + eps)
    df['시술_실패_횟수'] = (df['총 시술 횟수'] - df['총 임신 횟수']).clip(lower=0)
    egg_count = df['수집된 신선 난자 수'].fillna(-1)
    df['최적_난자수']    = ((egg_count >= 5) & (egg_count <= 15)).astype(int)
    df['나이_이식배아수'] = df['시술 당시 나이'] * df['이식된 배아 수'].fillna(0)
    return df

# ── 공통 전처리 ──
train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)
X_base_tr, X_base_te = preprocess(train_fe, test_fe)

# FE-v1 (EXP-020, EXP-028, EXP-032 LGB용)
X_v1_train = add_derived_features_v1(X_base_tr)
X_v1_test  = add_derived_features_v1(X_base_te)

# FE-v2 (EXP-032 CAT/XGB 베이스)
X_v2_train = add_derived_features_v2(X_base_tr)
X_v2_test  = add_derived_features_v2(X_base_te)

y_train = train[TARGET]
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()

# ── EXP-032 TE/ITE 설정 ──
TE_COLS = ['시술 시기 코드','시술 유형','특정 시술 유형','배란 유도 유형',
           '난자 출처','정자 출처','시술 당시 나이','총 시술 횟수','총 임신 횟수']
ITE_PAIRS = [('시술 당시 나이','시술 유형'),('시술 당시 나이','특정 시술 유형'),
             ('시술 당시 나이','난자 출처'),('시술 당시 나이','배란 유도 유형'),
             ('시술 유형','총 시술 횟수'),('난자 출처','시술 유형')]
TE_COLS   = [c for c in TE_COLS   if c in X_v2_train.columns]
ITE_PAIRS = [(c1,c2) for c1,c2 in ITE_PAIRS if c1 in X_v2_train.columns and c2 in X_v2_train.columns]
TE_ALPHA, ITE_ALPHA = 10, 20

print(f'FE-v1: {X_v1_train.shape[1]}개  /  FE-v2: {X_v2_train.shape[1]}개')
print(f'TE 컬럼: {len(TE_COLS)}개  /  ITE 쌍: {len(ITE_PAIRS)}개')

FE-v1: 85개  /  FE-v2: 89개
TE 컬럼: 9개  /  ITE 쌍: 6개


## 공통 모델 파라미터

In [3]:
LGB_PARAMS_BASE = dict(
    objective='binary', metric='auc', verbosity=-1,
    num_threads=-1, is_unbalance=True, bagging_freq=1,
    learning_rate=0.035425966303284824,
    num_leaves=266, max_depth=5, min_child_samples=166,
    feature_fraction=0.5346439126449233,
    bagging_fraction=0.7122309235479091,
    lambda_l1=9.901034988600228,
    lambda_l2=2.213951873239442,
    min_gain_to_split=0.11418176854933762,
)
CAT_PARAMS_BASE = dict(
    iterations=2000, loss_function='Logloss', eval_metric='AUC',
    auto_class_weights='Balanced', thread_count=-1, verbose=False,
    early_stopping_rounds=100,
    learning_rate=0.018758723768855998, depth=6,
    l2_leaf_reg=9.189608434163782, min_data_in_leaf=19,
    subsample=0.8170921295501483, colsample_bylevel=0.6936810336930781,
)
XGB_PARAMS_BASE = dict(
    n_estimators=2000, scale_pos_weight=neg_pos_ratio,
    eval_metric='auc', tree_method='hist',
    n_jobs=-1, verbosity=0, early_stopping_rounds=100,
    learning_rate=0.05520069867907647, max_depth=4, min_child_weight=59,
    subsample=0.7663066457187595, colsample_bytree=0.6581836436885355,
    reg_alpha=8.692038126211928, reg_lambda=0.23932562420374562,
)

def rank_avg_normalize(arrays):
    """여러 예측값 Rank Average → 0~1 정규화"""
    ranks = np.stack([rankdata(a) for a in arrays], axis=1)
    avg = ranks.mean(axis=1)
    return (avg - avg.min()) / (avg.max() - avg.min())

print('파라미터 설정 완료')

파라미터 설정 완료


## Part A — EXP-020: LGB+CAT+XGB, seed=42, FE-v1

In [4]:
print('=' * 60)
print('Part A: EXP-020  (seed=42, FE-v1)')
print('=' * 60)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
lgb_p = {**LGB_PARAMS_BASE, 'seed': SEED}
cat_p = {**CAT_PARAMS_BASE, 'random_seed': SEED}
xgb_p = {**XGB_PARAMS_BASE, 'random_state': SEED}

oof_a_lgb  = np.zeros(len(X_v1_train))
oof_a_cat  = np.zeros(len(X_v1_train))
oof_a_xgb  = np.zeros(len(X_v1_train))
test_a_lgb = np.zeros(len(X_v1_test))
test_a_cat = np.zeros(len(X_v1_test))
test_a_xgb = np.zeros(len(X_v1_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_v1_train, y_train), 1):
    X_tr, X_val = X_v1_train.iloc[tr_idx], X_v1_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    ds_tr = lgb.Dataset(X_tr, label=y_tr)
    ds_val = lgb.Dataset(X_val, label=y_val, reference=ds_tr)
    m = lgb.train(lgb_p, ds_tr, num_boost_round=2000, valid_sets=[ds_val],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
    oof_a_lgb[val_idx] = m.predict(X_val)
    test_a_lgb        += m.predict(X_v1_test) / N_FOLDS

    m = CatBoostClassifier(**cat_p)
    m.fit(X_tr, y_tr, eval_set=Pool(X_val, y_val), use_best_model=True)
    oof_a_cat[val_idx] = m.predict_proba(X_val)[:, 1]
    test_a_cat        += m.predict_proba(X_v1_test)[:, 1] / N_FOLDS

    m = xgb.XGBClassifier(**xgb_p)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_a_xgb[val_idx] = m.predict_proba(X_val)[:, 1]
    test_a_xgb        += m.predict_proba(X_v1_test)[:, 1] / N_FOLDS

auc_a = roc_auc_score(y_train, rank_avg_normalize([oof_a_lgb, oof_a_cat, oof_a_xgb]))
test_a = rank_avg_normalize([test_a_lgb, test_a_cat, test_a_xgb])
print(f'[A] OOF AUC (Rank Avg): {auc_a:.5f}')

Part A: EXP-020  (seed=42, FE-v1)
[A] OOF AUC (Rank Avg): 0.74068


## Part B — EXP-028: LGB+CAT+XGB, 5 seeds, FE-v1

In [5]:
print('=' * 60)
print('Part B: EXP-028  (5 seeds, FE-v1)')
print('=' * 60)

SEEDS = [42, 0, 123, 2024, 777]
oof_b_lgb  = np.zeros(len(X_v1_train))
oof_b_cat  = np.zeros(len(X_v1_train))
oof_b_xgb  = np.zeros(len(X_v1_train))
test_b_lgb = np.zeros(len(X_v1_test))
test_b_cat = np.zeros(len(X_v1_test))
test_b_xgb = np.zeros(len(X_v1_test))

for s_idx, seed in enumerate(SEEDS, 1):
    print(f'  Seed {seed} ({s_idx}/{len(SEEDS)})')
    lgb_p = {**LGB_PARAMS_BASE, 'seed': seed}
    cat_p = {**CAT_PARAMS_BASE, 'random_seed': seed}
    xgb_p = {**XGB_PARAMS_BASE, 'random_state': seed}
    skf   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    oof_s_lgb = np.zeros(len(X_v1_train))
    oof_s_cat = np.zeros(len(X_v1_train))
    oof_s_xgb = np.zeros(len(X_v1_train))
    test_s_lgb = np.zeros(len(X_v1_test))
    test_s_cat = np.zeros(len(X_v1_test))
    test_s_xgb = np.zeros(len(X_v1_test))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_v1_train, y_train), 1):
        X_tr, X_val = X_v1_train.iloc[tr_idx], X_v1_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        ds_tr = lgb.Dataset(X_tr, label=y_tr)
        ds_val = lgb.Dataset(X_val, label=y_val, reference=ds_tr)
        m = lgb.train(lgb_p, ds_tr, num_boost_round=2000, valid_sets=[ds_val],
                      callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
        oof_s_lgb[val_idx] = m.predict(X_val)
        test_s_lgb        += m.predict(X_v1_test) / N_FOLDS

        m = CatBoostClassifier(**cat_p)
        m.fit(X_tr, y_tr, eval_set=Pool(X_val, y_val), use_best_model=True)
        oof_s_cat[val_idx] = m.predict_proba(X_val)[:, 1]
        test_s_cat        += m.predict_proba(X_v1_test)[:, 1] / N_FOLDS

        m = xgb.XGBClassifier(**xgb_p)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        oof_s_xgb[val_idx] = m.predict_proba(X_val)[:, 1]
        test_s_xgb        += m.predict_proba(X_v1_test)[:, 1] / N_FOLDS

    oof_b_lgb  += oof_s_lgb  / len(SEEDS)
    oof_b_cat  += oof_s_cat  / len(SEEDS)
    oof_b_xgb  += oof_s_xgb  / len(SEEDS)
    test_b_lgb += test_s_lgb / len(SEEDS)
    test_b_cat += test_s_cat / len(SEEDS)
    test_b_xgb += test_s_xgb / len(SEEDS)

auc_b = roc_auc_score(y_train, rank_avg_normalize([oof_b_lgb, oof_b_cat, oof_b_xgb]))
test_b = rank_avg_normalize([test_b_lgb, test_b_cat, test_b_xgb])
print(f'[B] OOF AUC (Rank Avg): {auc_b:.5f}')

Part B: EXP-028  (5 seeds, FE-v1)
  Seed 42 (1/5)
  Seed 0 (2/5)
  Seed 123 (3/5)
  Seed 2024 (4/5)
  Seed 777 (5/5)
[B] OOF AUC (Rank Avg): 0.74081


## Part C — EXP-032: LGB(FE-v1) + CAT(FE-v2+TE) + XGB(FE-v2+TE+ITE)

In [6]:
print('=' * 60)
print('Part C: EXP-032  (mixed FE, seed=42)')
print('=' * 60)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
lgb_p = {**LGB_PARAMS_BASE, 'seed': SEED}
cat_p = {**CAT_PARAMS_BASE, 'random_seed': SEED}
xgb_p = {**XGB_PARAMS_BASE, 'random_state': SEED}

oof_c_lgb  = np.zeros(len(X_v1_train))
oof_c_cat  = np.zeros(len(X_v1_train))
oof_c_xgb  = np.zeros(len(X_v1_train))
test_c_lgb = np.zeros(len(X_v1_test))
test_c_cat = np.zeros(len(X_v1_test))
test_c_xgb = np.zeros(len(X_v1_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_v1_train, y_train), 1):
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    global_mean = y_tr.mean()

    # LGB: FE-v1
    X_tr_lgb  = X_v1_train.iloc[tr_idx]
    X_val_lgb = X_v1_train.iloc[val_idx]
    ds_tr  = lgb.Dataset(X_tr_lgb, label=y_tr)
    ds_val = lgb.Dataset(X_val_lgb, label=y_val, reference=ds_tr)
    m = lgb.train(lgb_p, ds_tr, num_boost_round=2000, valid_sets=[ds_val],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
    oof_c_lgb[val_idx] = m.predict(X_val_lgb)
    test_c_lgb        += m.predict(X_v1_test) / N_FOLDS

    # CAT: FE-v2 + TE (fold 내부에서 계산 — data leakage 방지)
    X_tr_cat  = X_v2_train.iloc[tr_idx].copy()
    X_val_cat = X_v2_train.iloc[val_idx].copy()
    X_te_cat  = X_v2_test.copy()
    for col in TE_COLS:
        grp = y_tr.groupby(X_tr_cat[col])
        smoothed = (grp.count() * grp.mean() + TE_ALPHA * global_mean) / (grp.count() + TE_ALPHA)
        te_col = f'{col}_te'
        X_tr_cat[te_col]  = X_tr_cat[col].map(smoothed).fillna(global_mean)
        X_val_cat[te_col] = X_val_cat[col].map(smoothed).fillna(global_mean)
        X_te_cat[te_col]  = X_te_cat[col].map(smoothed).fillna(global_mean)
    m = CatBoostClassifier(**cat_p)
    m.fit(X_tr_cat, y_tr, eval_set=Pool(X_val_cat, y_val), use_best_model=True)
    oof_c_cat[val_idx] = m.predict_proba(X_val_cat)[:, 1]
    test_c_cat        += m.predict_proba(X_te_cat)[:, 1] / N_FOLDS

    # XGB: FE-v2 + TE + ITE (fold 내부에서 계산)
    X_tr_xgb  = X_tr_cat.copy()
    X_val_xgb = X_val_cat.copy()
    X_te_xgb  = X_te_cat.copy()
    for col1, col2 in ITE_PAIRS:
        key_tr  = X_tr_xgb[col1].astype(str)  + '_' + X_tr_xgb[col2].astype(str)
        key_val = X_val_xgb[col1].astype(str) + '_' + X_val_xgb[col2].astype(str)
        key_te  = X_te_xgb[col1].astype(str)  + '_' + X_te_xgb[col2].astype(str)
        grp = y_tr.groupby(key_tr)
        smoothed = (grp.count() * grp.mean() + ITE_ALPHA * global_mean) / (grp.count() + ITE_ALPHA)
        ite_col = f'{col1}_{col2}_ite'
        X_tr_xgb[ite_col]  = key_tr.map(smoothed).fillna(global_mean)
        X_val_xgb[ite_col] = key_val.map(smoothed).fillna(global_mean)
        X_te_xgb[ite_col]  = key_te.map(smoothed).fillna(global_mean)
    m = xgb.XGBClassifier(**xgb_p)
    m.fit(X_tr_xgb, y_tr, eval_set=[(X_val_xgb, y_val)], verbose=False)
    oof_c_xgb[val_idx] = m.predict_proba(X_val_xgb)[:, 1]
    test_c_xgb        += m.predict_proba(X_te_xgb)[:, 1] / N_FOLDS

    fold_aucs = [roc_auc_score(y_val, oof_c_lgb[val_idx]),
                 roc_auc_score(y_val, oof_c_cat[val_idx]),
                 roc_auc_score(y_val, oof_c_xgb[val_idx])]
    print(f'  Fold {fold}  LGB={fold_aucs[0]:.5f}  CAT={fold_aucs[1]:.5f}  XGB={fold_aucs[2]:.5f}')

auc_c = roc_auc_score(y_train, rank_avg_normalize([oof_c_lgb, oof_c_cat, oof_c_xgb]))
test_c = rank_avg_normalize([test_c_lgb, test_c_cat, test_c_xgb])
print(f'[C] OOF AUC (Rank Avg): {auc_c:.5f}')

Part C: EXP-032  (mixed FE, seed=42)
  Fold 1  LGB=0.73812  CAT=0.73827  XGB=0.73833
  Fold 2  LGB=0.74318  CAT=0.74338  XGB=0.74301
  Fold 3  LGB=0.74072  CAT=0.74020  XGB=0.74036
  Fold 4  LGB=0.73880  CAT=0.73864  XGB=0.73893
  Fold 5  LGB=0.74160  CAT=0.74062  XGB=0.74156
[C] OOF AUC (Rank Avg): 0.74071


## Final — Rank Average(A, B, C) → Submission

In [7]:
# OOF 앙상블 점수
oof_final  = rank_avg_normalize([oof_a_lgb, oof_a_cat, oof_a_xgb,
                                  oof_b_lgb, oof_b_cat, oof_b_xgb,
                                  oof_c_lgb, oof_c_cat, oof_c_xgb])
# test 앙상블: A, B, C 각각의 Rank Avg 결과를 다시 Rank Avg
test_final = rank_avg_normalize([test_a, test_b, test_c])

auc_final = roc_auc_score(y_train, oof_final)
print(f'[A] OOF: {auc_a:.5f}')
print(f'[B] OOF: {auc_b:.5f}')
print(f'[C] OOF: {auc_c:.5f}')
print(f'[Final] OOF Rank Avg(A,B,C): {auc_final:.5f}')
print(f'probability 범위: [{test_final.min():.4f}, {test_final.max():.4f}]')

OUT_DIR.mkdir(parents=True, exist_ok=True)
sub['probability'] = test_final
auc_str   = f'{auc_final:.4f}'.replace('.', '')
out_fname = f'submission_final_{AUTHOR}_{auc_str}.csv'
sub.to_csv(OUT_DIR / out_fname, index=False)
print(f'저장: {OUT_DIR / out_fname}')

[A] OOF: 0.74068
[B] OOF: 0.74081
[C] OOF: 0.74071
[Final] OOF Rank Avg(A,B,C): 0.74081
probability 범위: [0.0000, 1.0000]
저장: ../data/submissions/submission_final_조여진_07408.csv


## 실험 기록

In [8]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5), round(recall,5), round(precision,5), round(accuracy,5), round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin   = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill   = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font   = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-034 기록 완료 (row {next_row})')

oof_binary = (oof_final >= 0.5).astype(int)
params_str = ('RankAvg(EXP-020+EXP-028+EXP-032) | '
              'A:LGB+CAT+XGB(seed=42,FE-v1) + B:LGB+CAT+XGB(5seeds,FE-v1) + C:LGB(FE-v1)+CAT(FE-v2+TE)+XGB(FE-v2+TE+ITE)')
NOTES    = '대회 코드 제출용 통합 노트북. EXP-020+028+032 파이프라인 통합 + Rank Average 최종 앙상블.'
INSIGHTS = (f'A(EXP-020)={auc_a:.5f}  B(EXP-028)={auc_b:.5f}  C(EXP-032)={auc_c:.5f}  '
            f'Final={auc_final:.5f}')

log_to_leaderboard(
    34, AUTHOR, 'Unified(EXP-020+028+032)', params_str,
    f1_score(y_train, oof_binary), recall_score(y_train, oof_binary),
    precision_score(y_train, oof_binary), accuracy_score(y_train, oof_binary),
    auc_final, f'Stratified {N_FOLDS}-Fold', 'mixed(v1/v2/TE/ITE)', X_v1_train.shape[1],
    'is_unbalance / Balanced / scale_pos_weight',
    'N', None, 'notebooks/30_final_code_submission_yjcho.ipynb', NOTES, INSIGHTS
)

[leaderboard.xlsx] EXP-034 기록 완료 (row 31)
